## SC-JEPA on log data

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import copy
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler

import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset

from data.utils import set_seed
from data.datasets import PredictiveMaintenanceDataset

from models.encoder import Encoder
from models.decoder import Decoder
from models.quantizer import Quantizer
from models.classifier import SimpleClassifier, FocalLoss
from models.predictor import TransformerPredictor, CoarsePredictor

from utils.evaluation import evaluate
from utils.evaluation import test_evaluation

### Pre-processing

In [3]:
def load_and_merge_pdm_data(telemetry_path, machines_path, errors_path, maint_path, failures_path):

    print("Loading files...")
    telemetry = pd.read_csv('/home/irene/deep_adv/sc_jepa_ver3/csv files/PdM_telemetry.csv')
    machines = pd.read_csv('/home/irene/deep_adv/sc_jepa_ver3/csv files/PdM_machines.csv')
    errors = pd.read_csv('/home/irene/deep_adv/sc_jepa_ver3/csv files/PdM_errors.csv')
    maint = pd.read_csv('/home/irene/deep_adv/sc_jepa_ver3/csv files/PdM_maint.csv')
    failures = pd.read_csv('/home/irene/deep_adv/sc_jepa_ver3/csv files/PdM_failures.csv')
    
    # Conversion in datetime format
    print("Converting dates...")
    for df in [telemetry, errors, maint, failures]:
        df['datetime'] = pd.to_datetime(df['datetime'])
        
    # Event Transformation
    # The events are categorical. We use One-Hot Encoding to transform them into numeric columns (1 or 0)
    # Grouping by 'datetime' and 'machineID' by summing them, in case of more events in the same hour
    
    print("Processing discrete events...")
    # Errors
    errors_dummy = pd.get_dummies(errors, columns=['errorID'], prefix='err', dtype=int)
    errors_grouped = errors_dummy.groupby(['datetime', 'machineID']).sum().reset_index()
    
    # Maintenance
    maint_dummy = pd.get_dummies(maint, columns=['comp'], prefix='maint', dtype=int)
    maint_grouped = maint_dummy.groupby(['datetime', 'machineID']).sum().reset_index()
    
    # Failures (our labels for the task)
    failures_dummy = pd.get_dummies(failures, columns=['failure'], prefix='fail', dtype=int)
    failures_grouped = failures_dummy.groupby(['datetime', 'machineID']).sum().reset_index()
    
    # Merging the data
    print("Merging...")
    df_merged = pd.merge(telemetry, machines, on='machineID', how='left')
    df_merged = pd.merge(df_merged, errors_grouped, on=['datetime', 'machineID'], how='left')
    df_merged = pd.merge(df_merged, maint_grouped, on=['datetime', 'machineID'], how='left')
    df_merged = pd.merge(df_merged, failures_grouped, on=['datetime', 'machineID'], how='left')
    
    # Cleaning and Null values handling
    print("Handling null values and sorting...")
    event_cols = [c for c in df_merged.columns if c.startswith(('err_', 'maint_', 'fail_'))]
    df_merged[event_cols] = df_merged[event_cols].fillna(0).astype(int)
    df_merged['model'] = df_merged['model'].str.replace('model', '').astype(int)

    df_merged = df_merged.sort_values(by=['machineID', 'datetime']).reset_index(drop=True)
    
    print("Pre-processing complete!")
    return df_merged

if __name__ == "__main__":
    input_dir = "/home/irene/deep_adv/sc_jepa_ver3" 
    
    output_dir = '/home/irene/deep_adv/sc_jepa_ver3/data'
    output_filename = "pdm_merged.csv"
    output_path = os.path.join(output_dir, output_filename)

    os.makedirs(output_dir, exist_ok=True)

    telemetry_path = os.path.join(input_dir, "PdM_telemetry.csv")
    machines_path = os.path.join(input_dir, "PdM_machines.csv")
    errors_path = os.path.join(input_dir, "PdM_errors.csv")
    maint_path = os.path.join(input_dir, "PdM_maint.csv")
    failures_path = os.path.join(input_dir, "PdM_failures.csv")

    try:
        df_final = load_and_merge_pdm_data(
            telemetry_path=telemetry_path, 
            machines_path=machines_path, 
            errors_path=errors_path, 
            maint_path=maint_path, 
            failures_path=failures_path
        )
        
        print(f"\nSize of the final dataset: {df_final.shape}")
        val_start = pd.to_datetime("2015-09-01")
        test_start = pd.to_datetime("2015-11-01")

        df_train = df_final[df_final['datetime'] < val_start]
        df_val = df_final[(df_final['datetime'] >= val_start) & (df_final['datetime'] < test_start)]
        df_test = df_final[df_final['datetime'] >= test_start]

        print(f"Training set size: {df_train.shape}")
        print(f"Validation set size: {df_val.shape}")
        print(f"Test set size: {df_test.shape}")

        df_train.to_csv(os.path.join(output_dir, "train.csv"), index=False)
        df_val.to_csv(os.path.join(output_dir, "val.csv"), index=False)
        df_test.to_csv(os.path.join(output_dir, "test.csv"), index=False)
        
        print("train.csv, val.csv e test.csv files saved successfully.")
    
    except FileNotFoundError as e:
        print(f"\nERROR: File not found. Check directory.")
        print(f"Detail: {e}")

Loading files...
Converting dates...
Processing discrete events...
Merging...
Handling null values and sorting...
Pre-processing complete!

Size of the final dataset: (876100, 21)
Training set size: (582600, 21)
Validation set size: (146400, 21)
Test set size: (147100, 21)
train.csv, val.csv e test.csv files saved successfully.


### Global variables and models

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")
set_seed(42)

WINDOW_SIZE = 24  
PATCH_LEN = 6
NUM_PATCHES = (WINDOW_SIZE // 2) // PATCH_LEN
IN_CHANNELS = 6   
LATENT_DIM = 64     
NUM_CODES = 64      
CNN_H_DIM = 64          
NHEAD = 2           
NUM_TRANS_LAYERS = 2

encoder = Encoder(
    num_patches=NUM_PATCHES, patch_len=PATCH_LEN, latent_dim=LATENT_DIM, 
    cnn_h_dim=CNN_H_DIM, trans_nhead=NHEAD, trans_num_layers=NUM_TRANS_LAYERS, in_channels=IN_CHANNELS
).to(DEVICE)

quantizer = Quantizer(num_codes=NUM_CODES, embedding_dim=LATENT_DIM).to(DEVICE)

predictor = TransformerPredictor(
    num_codes=NUM_CODES, nhead=NHEAD, num_layers=NUM_TRANS_LAYERS, 
    hidden_dim=128, num_patches=NUM_PATCHES, latent_dim=LATENT_DIM
).to(DEVICE)

coarse_predictor = CoarsePredictor(
    num_codes=NUM_CODES, nhead=NHEAD, num_layers=NUM_TRANS_LAYERS, 
    hidden_dim=128, num_patches=NUM_PATCHES, latent_dim=LATENT_DIM
).to(DEVICE)

decoder = Decoder(latent_dim=LATENT_DIM, out_channels=IN_CHANNELS, patch_len=PATCH_LEN).to(DEVICE)

print("[INFO] Encoder, Quantizer, Predictors e Decoder successfully initialized!")

[INFO] Using device: cuda
[INFO] Encoder, Quantizer, Predictors e Decoder successfully initialized!


/home/irene/deep_adv/sc_jepa_ver3/models/predictor.py:47: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(
/home/irene/deep_adv/sc_jepa_ver3/models/predictor.py:135: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


### Dataset

In [7]:
DATA_DIR = "/home/irene/deep_adv/sc_jepa_ver3/data" 
train_csv = os.path.join(DATA_DIR, "train.csv")
val_csv = os.path.join(DATA_DIR, "val.csv")

WINDOW_SIZE = 24  
BATCH_SIZE_PRETRAIN = 256

print("[INFO] Preparing data for pre-training...")

train_ds_pretrain = PredictiveMaintenanceDataset(
    train_csv, window_size=WINDOW_SIZE, mode='pretrain'
)

val_ds_pretrain = PredictiveMaintenanceDataset(
    val_csv, window_size=WINDOW_SIZE, mode='pretrain', scaler=train_ds_pretrain.scaler
)

train_loader_pretrain = DataLoader(train_ds_pretrain, batch_size=BATCH_SIZE_PRETRAIN, shuffle=True, drop_last=True)
val_loader_pretrain = DataLoader(val_ds_pretrain, batch_size=BATCH_SIZE_PRETRAIN, shuffle=False, drop_last=True)

print(f"The pre-training data are ready! Train: {len(train_ds_pretrain)}, Val: {len(val_ds_pretrain)}")

[INFO] Preparing data for pre-training...
Loading data /home/irene/deep_adv/sc_jepa_ver3/data/train.csv...
Creating time sequences...
Loading data /home/irene/deep_adv/sc_jepa_ver3/data/val.csv...
Creating time sequences...
The pre-training data are ready! Train: 580300, Val: 144100


### Encoder training

In [ ]:
from data.utils import (
    instance_normalize,
    reverse_instance_normalize,
    update_ema,
    coarse_scale_to_patch
)

from utils.losses import (
    kl_loss_fine,
    kl_loss_coarse,
    mse_alignment_loss,
    vq_losses,
    entropy_losses
)
set_seed(42)

NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
EMA_DECAY = 0.996 

KL_FINE_WEIGHT = 1.0
KL_COARSE_WEIGHT = 0.5
MSE_WEIGHT = 0.1
BETA = 0.5
COMMITMENT_WEIGHT = 0.25
ENTROPY_SAMPLE_WEIGHT = 0.001
ENTROPY_BATCH_WEIGHT = 0.005
PRED_TEMP = 0.8
RECON_WEIGHT_START = 0.5
RECON_WEIGHT_END = 0.1

# Data loading
DATA_DIR = "/home/irene/deep_adv/sc_jepa_ver3/data"
train_csv = os.path.join(DATA_DIR, "train.csv")
val_csv = os.path.join(DATA_DIR, "val.csv")
BATCH_SIZE = 256
WINDOW_SIZE = 24

print("[INFO] Loading dataset...")
train_ds = PredictiveMaintenanceDataset(train_csv, window_size=WINDOW_SIZE, mode='pretrain')
val_ds = PredictiveMaintenanceDataset(val_csv, window_size=WINDOW_SIZE, mode='pretrain', scaler=train_ds.scaler)

train_loader_pretrain = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=4, pin_memory=True)
val_loader_pretrain = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=True, num_workers=4, pin_memory=True)
print(f"[INFO] Train batches: {len(train_loader_pretrain)}, Val batches: {len(val_loader_pretrain)}")

# EMA encoder
print("[INFO] Preparing EMA Encoder...")
# creating a copy of both the encoder and quantizer (encoder_tgt and quantizer_tgt) which will act as teachers to give the correct answers to the students (encoder and quantizer)
encoder_tgt = copy.deepcopy(encoder).eval()
quantizer_tgt = copy.deepcopy(quantizer).eval()

for p in encoder_tgt.parameters(): p.requires_grad = False
for p in quantizer_tgt.parameters(): p.requires_grad = False

optimizer = optim.Adam(
    list(encoder.parameters()) + list(quantizer.parameters()) + 
    list(predictor.parameters()) + list(coarse_predictor.parameters()) + list(decoder.parameters()),
    lr=LEARNING_RATE,
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

CHECKPOINT_DIR = "/home/irene/deep_adv/sc_jepa_ver3/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
best_val_loss = float('inf')
best_model_path = os.path.join(CHECKPOINT_DIR, "encoder.pth")
PATIENCE = 10
patience_counter = 0

print("[INFO] Starting pre-training...")

# Training loop

for epoch in range(NUM_EPOCHS):

    encoder.train()
    quantizer.train()
    predictor.train()
    coarse_predictor.train()
    decoder.train()

    total_train_loss = 0.0
    batch_count = 0
    
    for x_past, x_future in tqdm(train_loader_pretrain, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False):
        batch_count += 1

        # reads how many samples are in the batch and then reorganizes the raw temporal windows  by dividing them
        # into patches and converting them in 4 dimensions (Batch, NUM_PATCHES, PATCH_LEN, IN_CHANNELS)
        B = x_past.shape[0]
        x_past = x_past.view(B, NUM_PATCHES, PATCH_LEN, IN_CHANNELS)
        x_future = x_future.view(B, NUM_PATCHES, PATCH_LEN, IN_CHANNELS)

        # computing the percentage of completion of the epoch to gradually reduce the reconstruction error weight (from 50% to 10%)
        progress = batch_count / len(train_loader_pretrain)
        recon_weight = RECON_WEIGHT_START - (RECON_WEIGHT_START - RECON_WEIGHT_END) * progress
        
        x_past_norm, mean, std = instance_normalize(x_past)
        x_future_norm, _, _ = instance_normalize(x_future)
        x_future_coarse = coarse_scale_to_patch(x_future, add_patch_dim=True)

        x_past_norm, x_future_norm, x_future_coarse = x_past_norm.to(DEVICE), x_future_norm.to(DEVICE), x_future_coarse.to(DEVICE)
        mean, std = mean.to(DEVICE), std.to(DEVICE)

        # the normalized data are passed to the encoder to obtain h_past, which will then be fed to the quantizer to output the 
        # probability (p_past) and the discrete latent vector (z_q_past)
        h_past = encoder(x_past_norm)
        p_past, z_q_past = quantizer(h_past)

        # the decoder tires to redefine the original signal (x_recon) starting from the past quantized vector
        # mean and std are re-added
        # the result is compared with the real x_past by computing the reconstruction loss
        x_recon = decoder(z_q_past)
        x_recon = reverse_instance_normalize(x_recon, mean, std)
        x_target = x_past.permute(0, 1, 3, 2).to(DEVICE)
        
        loss_recon = torch.nn.functional.mse_loss(x_recon, x_target)

        # the predictor tries to guess the future starting from the past.
        # the target encoder sees the real future and calculates the real quantized target (p_future)
        logits_pred, z_pred = predictor(p_past)
        h_future = encoder_tgt(x_future_norm)
        p_future, z_q_future = quantizer_tgt(h_future)
        loss_kl = kl_loss_fine(logits_pred, p_future, PRED_TEMP)

        # same logic, but for the macro-scale. The predictor (coarse_predictor) tries to predict the blurred future
        # and is penalized if it makes mistakes
        logits_coarse, _ = coarse_predictor(p_past)
        h_future_c = encoder_tgt(instance_normalize(x_future_coarse)[0])
        p_future_c, _ = quantizer_tgt(h_future_c)
        loss_kl_c = kl_loss_coarse(logits_coarse, p_future_c, PRED_TEMP)

        # regularization losses
        loss_mse = mse_alignment_loss(z_pred, z_q_future)
        loss_q, loss_commit = vq_losses(h_past, z_q_past)
        loss_ent_s, loss_ent_b = entropy_losses(p_past)

        loss = (KL_FINE_WEIGHT * loss_kl + KL_COARSE_WEIGHT * loss_kl_c + 
                MSE_WEIGHT * loss_mse + BETA * loss_q + 
                COMMITMENT_WEIGHT * loss_commit + ENTROPY_SAMPLE_WEIGHT * loss_ent_s + 
                ENTROPY_BATCH_WEIGHT * loss_ent_b + recon_weight * loss_recon)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(encoder.parameters()) + list(predictor.parameters()), 
            max_norm=1.0
        )
        optimizer.step()

        update_ema(encoder, encoder_tgt, EMA_DECAY)
        update_ema(quantizer, quantizer_tgt, EMA_DECAY)
        
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader_pretrain)

    # Validation loop
    encoder.eval()
    quantizer.eval()
    predictor.eval()
    coarse_predictor.eval()
    decoder.eval()
    
    total_val_loss = 0.0
    
    with torch.no_grad():
        for x_past, x_future in val_loader_pretrain:
            B = x_past.shape[0]
            x_past = x_past.view(B, NUM_PATCHES, PATCH_LEN, IN_CHANNELS)
            x_future = x_future.view(B, NUM_PATCHES, PATCH_LEN, IN_CHANNELS)

            x_past_norm, mean, std = instance_normalize(x_past)
            x_future_norm, _, _ = instance_normalize(x_future)
            x_future_coarse = coarse_scale_to_patch(x_future, add_patch_dim=True)

            x_past_norm, x_future_norm, x_future_coarse = x_past_norm.to(DEVICE), x_future_norm.to(DEVICE), x_future_coarse.to(DEVICE)
            mean, std = mean.to(DEVICE), std.to(DEVICE)

            h_past = encoder(x_past_norm)
            p_past, z_q_past = quantizer(h_past)
            x_recon = decoder(z_q_past)
            x_recon = reverse_instance_normalize(x_recon, mean, std)
            x_target = x_past.permute(0, 1, 3, 2).to(DEVICE)
            
            loss_recon = torch.nn.functional.mse_loss(x_recon, x_target)

            logits_pred, z_pred = predictor(p_past)
            h_future = encoder_tgt(x_future_norm)
            p_future, z_q_future = quantizer_tgt(h_future)
            loss_kl = kl_loss_fine(logits_pred, p_future, PRED_TEMP)

            logits_coarse, _ = coarse_predictor(p_past)
            h_future_c = encoder_tgt(instance_normalize(x_future_coarse)[0])
            p_future_c, _ = quantizer_tgt(h_future_c)
            loss_kl_c = kl_loss_coarse(logits_coarse, p_future_c, PRED_TEMP)

            loss_mse = mse_alignment_loss(z_pred, z_q_future)
            loss_q, loss_commit = vq_losses(h_past, z_q_past)
            loss_ent_s, loss_ent_b = entropy_losses(p_past)

            val_loss = (KL_FINE_WEIGHT * loss_kl + KL_COARSE_WEIGHT * loss_kl_c + 
                        MSE_WEIGHT * loss_mse + BETA * loss_q + 
                        COMMITMENT_WEIGHT * loss_commit + ENTROPY_SAMPLE_WEIGHT * loss_ent_s + 
                        ENTROPY_BATCH_WEIGHT * loss_ent_b + RECON_WEIGHT_END * loss_recon)
            
            total_val_loss += val_loss.item()
            
    avg_val_loss = total_val_loss / len(val_loader_pretrain)
    
    print(f"[EPOCH {epoch+1}/{NUM_EPOCHS}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        print(f"  --> Model improved! Saving weights in {best_model_path}")
        torch.save(encoder.state_dict(), best_model_path) 
        
    else:
        patience_counter += 1
        print(f"  --> No improvement. Patience: {patience_counter}/{PATIENCE}")
        
        if patience_counter >= PATIENCE:
            print(f"\n[EARLY STOPPING] Stopped at epoch {epoch+1}. The validation loss has not improved for {PATIENCE} epochs.")
            break

print("\n[INFO] Pre-training complete!")

[INFO] Loading dataset...
Loading data /home/irene/deep_adv/sc_jepa_ver3/data/train.csv...
Creating time sequences...
Loading data /home/irene/deep_adv/sc_jepa_ver3/data/val.csv...
Creating time sequences...
[INFO] Train batches: 2266, Val batches: 562
[INFO] Preparing EMA Encoder...
[INFO] Starting pre-training...


[EPOCH 1/50] Train Loss: 0.2278 | Val Loss: 0.1466
  --> Model improved! Saving weights in /home/irene/deep_adv/sc_jepa_ver3/checkpoints/encoder.pth


[EPOCH 2/50] Train Loss: 0.2337 | Val Loss: 0.1491
  --> No improvement. Patience: 1/10


[EPOCH 3/50] Train Loss: 0.2212 | Val Loss: 0.1321
  --> Model improved! Saving weights in /home/irene/deep_adv/sc_jepa_ver3/checkpoints/encoder.pth


[EPOCH 4/50] Train Loss: 0.2078 | Val Loss: 0.1269
  --> Model improved! Saving weights in /home/irene/deep_adv/sc_jepa_ver3/checkpoints/encoder.pth


...

[EPOCH 46/50] Train Loss: 0.1545 | Val Loss: 0.0924
  --> Model improved! Saving weights in /home/irene/deep_adv/sc_jepa_ver3/checkpoints/encoder.pth


[EPOCH 47/50] Train Loss: 0.1547 | Val Loss: 0.0924
  --> Model improved! Saving weights in /home/irene/deep_adv/sc_jepa_ver3/checkpoints/encoder.pth


[EPOCH 48/50] Train Loss: 0.1543 | Val Loss: 0.0924
  --> Model improved! Saving weights in /home/irene/deep_adv/sc_jepa_ver3/checkpoints/encoder.pth


[EPOCH 49/50] Train Loss: 0.1541 | Val Loss: 0.0914
  --> Model improved! Saving weights in /home/irene/deep_adv/sc_jepa_ver3/checkpoints/encoder.pth


[EPOCH 50/50] Train Loss: 0.1541 | Val Loss: 0.0924
  --> No improvement. Patience: 1/10

[INFO] Pre-training complete!


### Data preparation for downstream

In [8]:
test_csv = os.path.join(DATA_DIR, "test.csv")
BATCH_SIZE_DOWNSTREAM = 256

print("[INFO] Preparing data for downstream classification...")

train_ds_downstream = PredictiveMaintenanceDataset(
    train_csv, window_size=WINDOW_SIZE, mode='downstream'
)

val_ds_downstream = PredictiveMaintenanceDataset(
    val_csv, window_size=WINDOW_SIZE, mode='downstream', scaler=train_ds_downstream.scaler
)

test_ds_downstream = PredictiveMaintenanceDataset(
    test_csv, window_size=WINDOW_SIZE, mode='downstream', scaler=train_ds_downstream.scaler
)

# Nota: in run_downstream.py applicavamo anche un "DownstreamWrapper" 
# e il "WeightedRandomSampler" per bilanciare le classi. Quelli andranno inseriti 
# direttamente qui o nella cella successiva prima del DataLoader!

print(f"Downstream data ready! Train: {len(train_ds_downstream)}, Val: {len(val_ds_downstream)}, Test: {len(test_ds_downstream)}")

[INFO] Preparing data for downstream classification...
Loading data /home/irene/deep_adv/sc_jepa_ver3/data/train.csv...
Creating time sequences...
Loading data /home/irene/deep_adv/sc_jepa_ver3/data/val.csv...
Creating time sequences...
Loading data /home/irene/deep_adv/sc_jepa_ver3/data/test.csv...
Creating time sequences...
Downstream data ready! Train: 579100, Val: 142900, Test: 143600


### Downstream training

In [ ]:
set_seed(42)

class DownstreamWrapper(Dataset):
    def __init__(self, original_ds, num_patches, patch_len, in_channels):
        self.ds = original_ds
        self.num_patches = num_patches
        self.patch_len = patch_len
        self.in_channels = in_channels
        
        print("[INFO] Extracting labels...")
        y_matrix = self.ds.df[self.ds.label_cols].values
        labels = []
        
        for seq in self.ds.sequences:
            y_window = y_matrix[seq['label_start'] : seq['label_end']]                  # isolating only the labels of the current temporal window
            y_label = np.zeros(1) if y_window.size == 0 else np.max(y_window, axis=0)
            y_tensor = torch.tensor(y_label, dtype=torch.float32)
            y_expanded = y_tensor.repeat(self.num_patches).to(torch.long)               # creating a tensor and repeating the same process for every patch
            labels.append(y_expanded)
            
        self.y_label = torch.stack(labels)  # merging all lables in the same tensor

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        """
        - Extracting the features from the original dataset
        - Calculating the number of temporal steps are needed
        - Cutting the sequence by taking only the recent data required to fill the patch
        - Remodeling the 2D matrix in a 3D tensor (that is the format sc-jepa expects)
        """
        x, _ = self.ds[idx]
        y_expanded = self.y_label[idx]
        past_hours = self.num_patches * self.patch_len
        x_recent = x[-past_hours:, :]
        x_reshaped = x_recent.view(self.num_patches, self.patch_len, self.in_channels)
        return x_reshaped, y_expanded

print("[INFO] Applying Downstream Wrapper...")
print("[INFO] Loading raw data for Downstream...")
DATA_DIR = "./data/" 
train_csv = os.path.join(DATA_DIR, "train.csv")
val_csv = os.path.join(DATA_DIR, "val.csv")
test_csv = os.path.join(DATA_DIR, "test.csv")

train_ds_downstream = PredictiveMaintenanceDataset(train_csv, window_size=WINDOW_SIZE, mode='downstream')
val_ds_downstream = PredictiveMaintenanceDataset(val_csv, window_size=WINDOW_SIZE, mode='downstream', scaler=train_ds_downstream.scaler)
test_ds_downstream = PredictiveMaintenanceDataset(test_csv, window_size=WINDOW_SIZE, mode='downstream', scaler=train_ds_downstream.scaler)

train_ds_wrapped = DownstreamWrapper(train_ds_downstream, NUM_PATCHES, PATCH_LEN, IN_CHANNELS)
val_ds_wrapped = DownstreamWrapper(val_ds_downstream, NUM_PATCHES, PATCH_LEN, IN_CHANNELS)
test_ds_wrapped = DownstreamWrapper(test_ds_downstream, NUM_PATCHES, PATCH_LEN, IN_CHANNELS)

# Balancing classes
print("[INFO] Computing weights...")
y_train = train_ds_wrapped.y_label.max(dim=1).values.numpy()
class_counts = np.bincount(y_train.astype(int))
weights = 1.0 / class_counts                    # the least frequent class has a larger magnitude weight
samples_weights = weights[y_train.astype(int)]  # assigning weights
sampler = WeightedRandomSampler(weights=samples_weights, num_samples=len(samples_weights), replacement=True)

BATCH_SIZE = 256
train_loader_down = DataLoader(train_ds_wrapped, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True)
val_loader_down = DataLoader(val_ds_wrapped, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader_down = DataLoader(test_ds_wrapped, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print("[INFO] Loading encoder weights...")
best_model_path = '/home/irene/deep_adv/sc_jepa_ver3/checkpoints/encoder.pth'
encoder.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

classifier = SimpleClassifier(input_dim=LATENT_DIM, num_patches=NUM_PATCHES).to(DEVICE)
criterion = FocalLoss(gamma=2.0, weight=torch.tensor([1.0, 1.0], device=DEVICE))

optimizer = optim.AdamW([
    {'params': encoder.parameters(), 'lr': 5e-5},
    {'params': classifier.parameters(), 'lr': 1e-3}
], weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

# Training loop

EPOCHS = 50
best_auc = -1.0
best_thresh = 0.5
wait = 0
patience = 10
CHECKPOINT_DIR = '/home/irene/deep_adv/sc_jepa_ver3/checkpoints'
downstream_model_path = os.path.join(CHECKPOINT_DIR, "downstream.pth")

print("[INFO] Starting classifier training...")

for epoch in range(1, EPOCHS + 1):
    classifier.train()
    encoder.train()
    total_loss = 0
    
    for x_patch, y_label in tqdm(train_loader_down, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
        x_patch, y_label = x_patch.to(DEVICE), y_label.to(DEVICE)

        # 1. the data are fed to the encoder to extract latent representations
        # 2. the extracted features are fed to the classifier to get the logits
        feats = encoder(x_patch)
        logits = classifier(feats)
        
        y_win = y_label.max(dim=1).values   # going back to one value per temporal window (to compute the error using the classificator's predictions)
        loss = criterion(logits, y_win)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(classifier.parameters()) + list(encoder.parameters()), 1.0) # clipping the gradient to a maximum value of 1 to avoid exploding gradients
        optimizer.step()
        
        total_loss += loss.item()

    val_thresh, f1, _, val_auc, _ = evaluate(classifier, encoder, val_loader_down, return_metrics=True, device=DEVICE)
    scheduler.step(val_auc)
    
    print(f"Epoch {epoch} | Loss: {total_loss/len(train_loader_down):.4f} | Val F1: {f1:.4f} | Val AUC: {val_auc:.4f}")

    if val_auc > best_auc:
        best_auc = val_auc
        best_thresh = val_thresh
        wait = 0
        
        torch.save({
            "classifier_state_dict": classifier.state_dict(),
            "encoder_state_dict": encoder.state_dict(),
            "threshold": best_thresh,
        }, downstream_model_path) 
        print(f"   --> Saving the best model (AUC: {best_auc:.4f})")
    else:
        wait += 1

    if wait >= patience:
        print(f"[EARLY STOPPING] No improvement for {patience} epochs.")
        break

print(f"\n[INFO] Downstream Complete! Best threshold: {best_thresh:.4f}")

[INFO] Applying Downstream Wrapper...
[INFO] Loading raw data for Downstream...
Loading data ./data/train.csv...
Creating time sequences...
Loading data ./data/val.csv...
Creating time sequences...
Loading data ./data/test.csv...
Creating time sequences...
[INFO] Extracting labels...
[INFO] Extracting labels...
[INFO] Extracting labels...
[INFO] Computing weights...
[INFO] Loading encoder weights...
[INFO] Starting classifier training...


Epoch 1 | Loss: 0.1030 | Val F1: 0.1733 | Val AUC: 0.9477
   --> Saving the best model (AUC: 0.9477)


Epoch 2 | Loss: 0.0712 | Val F1: 0.1880 | Val AUC: 0.9563
   --> Saving the best model (AUC: 0.9563)


Epoch 3 | Loss: 0.0622 | Val F1: 0.1904 | Val AUC: 0.9598
   --> Saving the best model (AUC: 0.9598)


Epoch 4 | Loss: 0.0564 | Val F1: 0.1949 | Val AUC: 0.9615
   --> Saving the best model (AUC: 0.9615)


...

Epoch 22 | Loss: 0.0475 | Val F1: 0.2048 | Val AUC: 0.9620


Epoch 23 | Loss: 0.0476 | Val F1: 0.2042 | Val AUC: 0.9619


Epoch 24 | Loss: 0.0476 | Val F1: 0.2037 | Val AUC: 0.9617


Epoch 25 | Loss: 0.0471 | Val F1: 0.2067 | Val AUC: 0.9620


Epoch 26 | Loss: 0.0474 | Val F1: 0.2083 | Val AUC: 0.9619
[EARLY STOPPING] No improvement for 10 epochs.

[INFO] Downstream Complete! Best threshold: 0.7000


### Test

In [ ]:
set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WINDOW_SIZE = 24  
IN_CHANNELS = 6   
PATCH_LEN = 6
NUM_PATCHES = (WINDOW_SIZE // 2) // PATCH_LEN
LATENT_DIM = 64
BATCH_SIZE = 256

class DownstreamWrapper(Dataset):
    def __init__(self, original_ds, num_patches, patch_len, in_channels):
        self.ds = original_ds
        self.num_patches = num_patches
        self.patch_len = patch_len
        self.in_channels = in_channels
        
        y_matrix = self.ds.df[self.ds.label_cols].values
        labels = []
        for seq in self.ds.sequences:
            y_window = y_matrix[seq['label_start'] : seq['label_end']]
            y_label = np.zeros(1) if y_window.size == 0 else np.max(y_window, axis=0)
            y_expanded = torch.tensor(y_label, dtype=torch.float32).repeat(self.num_patches).to(torch.long)
            labels.append(y_expanded)
        self.y_label = torch.stack(labels)

    def __len__(self): return len(self.ds)

    def __getitem__(self, idx):
        x, _ = self.ds[idx]
        y_expanded = self.y_label[idx]
        x_recent = x[-(self.num_patches * self.patch_len):, :]
        return x_recent.view(self.num_patches, self.patch_len, self.in_channels), y_expanded

print("[INFO] Loading test data...")
DATA_DIR = "./data/"
train_csv = os.path.join(DATA_DIR, "train.csv")
test_csv = os.path.join(DATA_DIR, "test.csv")

train_ds_scaler = PredictiveMaintenanceDataset(train_csv, window_size=WINDOW_SIZE, mode='downstream')
test_ds_raw = PredictiveMaintenanceDataset(test_csv, window_size=WINDOW_SIZE, mode='downstream', scaler=train_ds_scaler.scaler)
test_ds_wrapped = DownstreamWrapper(test_ds_raw, NUM_PATCHES, PATCH_LEN, IN_CHANNELS)
test_loader_down = DataLoader(test_ds_wrapped, batch_size=BATCH_SIZE, shuffle=False, num_workers=8)

classifier = SimpleClassifier(input_dim=LATENT_DIM, num_patches=NUM_PATCHES).to(DEVICE)

print("\n" + "="*50)
print("Starting evaluation on test set...")
print("="*50)

downstream_model_path = '/home/irene/deep_adv/sc_jepa_ver3/checkpoints/downstream.pth'
print(f"[INFO] Loading weights from {downstream_model_path}...")
ckpt = torch.load(downstream_model_path, map_location=DEVICE, weights_only=False)

encoder.load_state_dict(ckpt["encoder_state_dict"])
classifier.load_state_dict(ckpt["classifier_state_dict"])
val_threshold = ckpt["threshold"]

encoder.eval()
classifier.eval()

print(f"[INFO] Using the optimal threshold computed during validation: {val_threshold:.4f}\n")
test_evaluation(classifier, encoder, test_loader_down, val_threshold, device=DEVICE)
print("\n[INFO] Evaluation Complete!")

[INFO] Loading test data...
Loading data ./data/train.csv...
Creating time sequences...
Loading data ./data/test.csv...
Creating time sequences...

Starting evaluation on test set...
[INFO] Loading weights from /home/irene/deep_adv/sc_jepa_ver3/checkpoints/downstream.pth...
[INFO] Using the optimal threshold computed during validation: 0.7000



Test Set Evaluation: 100%|██████████| 561/561 [00:42<00:00, 13.26it/s]


--- Test set results ---
ROC AUC Score: 0.9596
Threshold applied: 0.7000

Classification Report:
              precision    recall  f1-score   support

           0     0.9961    0.9573    0.9763    142283
           1     0.1146    0.5968    0.1923      1317

    accuracy                         0.9540    143600
   macro avg     0.5554    0.7771    0.5843    143600
weighted avg     0.9880    0.9540    0.9692    143600


Confusion Matrix:
True Negative (TN): 136213 | False Positive (FP): 6070
False Negative (FN): 531 | True Positive (TP): 786

[INFO] Evaluation Complete!
